# Unit 9 -- Direct shooting

**4SC000 Optimal Control and Reinforcement Learning**

Single and multiple shooting on the running example of Unit 9: the
minimum-effort swing-up of a torque-driven pendulum,

$$\dot x = \begin{bmatrix}\dot\theta\\ -\sin\theta + u\end{bmatrix},
\qquad x(0)=\begin{bmatrix}0\\0\end{bmatrix},
\qquad x(T)=\begin{bmatrix}\pi\\0\end{bmatrix},
\qquad \min \int_0^T u^2\,dt .$$

All the machinery lives in `shooting.py` next to this notebook; nothing is
re-implemented here. The notebook is for *looking at* what it does.

Three things are worth taking away, and the third is the one that matters
most in practice:

1. single shooting eliminates the states by simulation and optimizes over
   the controls alone -- exactly what Unit 7's static optimization did,
   generalized to nonlinear dynamics;
2. it becomes **ill-conditioned** as the horizon grows, and multiple
   shooting fixes that by cutting the horizon into segments;
3. which local optimum you land in is decided by the **initial guess**, not
   by the method.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

# the swing-up problem used throughout Unit 9
x0      = np.array([0.0, 0.0])      # hanging down, at rest
xtarget = np.array([np.pi, 0.0])    # inverted, at rest
T, h    = 6.0, 60                   # horizon and number of steps
tau     = T / h
t_grid  = np.arange(h + 1) * tau
print(f"h = {h} steps of tau = {tau:.3f} s over T = {T} s")

In [ ]:
import shooting

u_ss, xs_ss, effort_ss = shooting.single_shooting_solve(x0, xtarget, h, tau)
print(f"final state   theta = {xs_ss[-1,0]:.4f}  (target {np.pi:.4f})")
print(f"              thetadot = {xs_ss[-1,1]:+.4f}")
print(f"effort  tau*sum(u^2) = {effort_ss:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(t_grid, xs_ss[:, 0], lw=2)
ax[0].axhline(np.pi, color="0.5", ls=":")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$\theta(t)$"); ax[0].set_title("state")
ax[1].step(t_grid[:-1], u_ss, where="post", lw=2)
ax[1].set_xlabel("t"); ax[1].set_ylabel("u(t)"); ax[1].set_title("control")
plt.tight_layout(); plt.show()

## Why single shooting struggles over long horizons

Shooting works by propagating sensitivities: the Jacobian
$\partial x_h/\partial u$ is built by chaining the per-step Jacobians
$A_k = \partial f/\partial x$ all the way to the end of the horizon. Any
instability in the system is therefore *raised to the power of the number of
steps*, and the Jacobian's condition number grows with it. An ill-conditioned
Jacobian is one the optimizer cannot take a reliable step on.

`sensitivity_condition_number` measures exactly this. The conditioning
depends on the trajectory, so it has to be evaluated **at the solution** --
at a zero control the pendulum just hangs at its stable equilibrium and
nothing is amplified. The cell below therefore solves each horizon first.
(It takes a minute or so; the longest solve is the slow one.)

In [ ]:
S_diag = 6
Ts = [6.0, 9.0, 12.0]   # h = 60, 90, 120, all divisible by S_diag
cond_single, cond_multi = [], []
for T_ in Ts:
    h_ = int(10 * T_)
    tau_ = T_ / h_
    u_, _, _ = shooting.single_shooting_solve(x0, xtarget, h_, tau_)
    cond_single.append(shooting.sensitivity_condition_number(x0, u_, tau_))
    # and the same quantity for ONE segment of a multiple-shooting
    # discretization, where sensitivities only propagate h_/S steps
    _, _, zs_, _ = shooting.multiple_shooting_solve(x0, xtarget, h_, S_diag, tau_)
    m_ = h_ // S_diag
    _, _, dedu = shooting.simulate_segment(zs_[0], u_[:m_], tau_)
    sv = np.linalg.svd(dedu, compute_uv=False)
    cond_multi.append(sv[0] / sv[-1])
    print(f"  T = {T_:4.0f} s (h = {h_:3d})   single {cond_single[-1]:9.2f}   "
          f"per segment {cond_multi[-1]:7.2f}")

plt.figure(figsize=(6.4, 3.6))
plt.semilogy(Ts, cond_single, "o-", lw=1.8, label="single shooting")
plt.semilogy(Ts, cond_multi, "s-", lw=1.8, label=f"per segment ({S_diag} segments)")
plt.xlabel("horizon $T$ [s]"); plt.ylabel(r"condition number of $\partial x_h/\partial u$")
plt.legend(); plt.tight_layout(); plt.show()

## Multiple shooting

Cut the horizon into $S$ segments, give each one its own free initial state,
and require continuity between them as explicit constraints ("defects").
Sensitivities now only propagate within a segment, so the conditioning stays
mild -- at the price of more decision variables and more constraints.

In [ ]:
S = 6
u_ms, xs_ms, zs, effort_ms = shooting.multiple_shooting_solve(x0, xtarget, h, S, tau)
print(f"segments S = {S},  {h//S} steps each")
print(f"final state   theta = {xs_ms[-1,0]:.4f},  thetadot = {xs_ms[-1,1]:+.4f}")
print(f"effort = {effort_ms:.4f}   (single shooting: {effort_ss:.4f})")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(t_grid, xs_ss[:, 0], lw=2, label="single shooting")
ax[0].plot(t_grid, xs_ms[:, 0], lw=2, ls="--", label="multiple shooting")
for i, z in enumerate(zs):
    ax[0].plot((i * h // S) * tau, z[0], "ko", ms=5,
               label="segment starts" if i == 0 else None)
ax[0].axhline(np.pi, color="0.5", ls=":")
ax[0].set_xlabel("t"); ax[0].set_ylabel(r"$\theta(t)$"); ax[0].legend(fontsize=9)
ax[1].step(t_grid[:-1], u_ss, where="post", lw=2, label="single")
ax[1].step(t_grid[:-1], u_ms, where="post", lw=2, ls="--", label="multiple")
ax[1].set_xlabel("t"); ax[1].set_ylabel("u(t)"); ax[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## The part that actually decides the answer

The two trajectories above do not merely differ in accuracy -- they are
*different solutions*. Single shooting swings the pendulum **backwards**
first; multiple shooting goes straight up.

Neither is a defect of the method. The swing-up problem is nonconvex and has
several local optima, and the two solvers were handed different starting
points: `single_shooting_solve` starts from $u \equiv 0$, while
`multiple_shooting_solve` additionally seeds its interior states along the
straight line from $x(0)$ to $x(T)$ -- a guess that already "knows" the
pendulum is supposed to go up.

The experiment below settles it. Give single shooting a warm start that goes
up, and it finds the other optimum too -- and that one is cheaper.

In [ ]:
def single_shooting_from(u_init, rho=300.0, iters=80):
    """single_shooting_solve, but from a chosen initial guess."""
    def residual(u):
        xs, dxdu = shooting.sensitivity_final_state(x0, u, tau)
        R = np.concatenate([np.sqrt(tau) * u, np.sqrt(rho) * (xs[-1] - xtarget)])
        J = np.vstack([np.sqrt(tau) * np.eye(len(u)), np.sqrt(rho) * dxdu])
        return R, J
    u = shooting._levenberg_marquardt(residual, u_init, iters=iters)
    return u, shooting.simulate(x0, u, tau)

u_cold, xs_cold = single_shooting_from(np.zeros(h))
u_warm, xs_warm = single_shooting_from(u_ms)      # warm start from multiple shooting

for name, u_, xs_ in (("from u = 0", u_cold, xs_cold),
                      ("warm started", u_warm, xs_warm)):
    print(f"  {name:14s}  theta(T) = {xs_[-1,0]:.4f}   "
          f"min theta = {xs_[:,0].min():+.3f}   "
          f"effort = {tau*np.sum(u_**2):.4f}")

In [ ]:
plt.figure(figsize=(6.6, 3.6))
plt.plot(t_grid, xs_cold[:, 0], lw=2, label="single shooting from $u=0$")
plt.plot(t_grid, xs_warm[:, 0], lw=2, ls="--", label="single shooting, warm started")
plt.axhline(np.pi, color="0.5", ls=":")
plt.axhline(0.0, color="0.8", lw=0.8)
plt.xlabel("t"); plt.ylabel(r"$\theta(t)$")
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

Same code, same problem, same tolerances -- two answers, and the one you
get is chosen by the initial guess alone.

This is the single most important practical fact about every method in this
unit. None of them is a global optimizer. A good initial guess is not a
convenience, it is part of the problem specification; a common recipe is to
solve a coarse collocation problem first and warm-start everything else from
it.

Continue with `A_9_CollocationPendulum.ipynb`.